# 🐝 QuantHive — Multi-Agent GRPO Training Notebook

**Project:** QuantHive — Decentralized Multi-Agent Trading Governance  
**Model:** Qwen 2.5 1.5B (4-bit quantized via Unsloth)  
**Method:** GRPO (Group Relative Policy Optimization) with 5 independent reward verifiers  
**Framework:** PettingZoo AEC + OpenEnv + TRL + Unsloth  

This notebook trains the **Trader agent** to propose compliant, profitable trades while respecting governance constraints set by the **Risk Manager** and **Portfolio Manager** agents.

---

## Architecture

```
┌──────────────┐    constraints     ┌──────────────────┐   allocation   ┌─────────┐
│ Risk Manager │ ──────────────────►│ Portfolio Manager │──────────────►│ Trader  │
│ (rule-based) │   size_limit,      │   (rule-based)   │  cap_alloc,   │ (LLM)   │
│              │   allow_new,       │                  │  override     │ Qwen2.5 │
│              │   force_reduce     │                  │               │ 1.5B    │
└──────────────┘                    └──────────────────┘               └─────────┘
        ▲                                    ▲                             │
        │              PettingZoo AEC Turn Order                           │
        └──────────────── Market Feedback ◄────────────────────────────────┘
```

**5 Reward Verifiers:**
1. `format_reward_func` — Enforces `<thought>...</thought><action>...</action>` format
2. `alignment_reward_func` — Anti-hallucination: reasoning must match market signals
3. `risk_reward_func_multiagent` — Compliance with Risk Manager size limits
4. `profit_reward_func` — Directional correctness vs. price trend
5. `governance_reward_func_multiagent` — Full governance compliance (RM + PM limits)

## 1. Environment Setup

In [ ]:
# ─── Check GPU availability ─────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ GPU not available. Go to Runtime → Change runtime type → T4 GPU"
    )
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU: {gpu}  |  VRAM: {mem:.1f} GB")

In [ ]:
# ─── Install Unsloth (must come first — patches TRL internals) ──────────────
%pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps xformers trl peft accelerate bitsandbytes triton
%pip install datasets transformers sentencepiece protobuf
%pip install pettingzoo>=1.24.0 gymnasium pandas numpy matplotlib

In [ ]:
# ─── Clone the QuantHive repository ──────────────────────────────────────────
import os

REPO_URL = "https://github.com/ARKAISW/multi-agent-trading-env.git"
REPO_DIR = "/content/multi-agent-trading-env"

if os.path.exists(REPO_DIR):
    print(f"📁 Repo already cloned at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f"✅ Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

In [ ]:
# ─── Add repo root to Python path so all imports work ────────────────────────
import sys
sys.path.insert(0, REPO_DIR)

# Verify critical imports
from env.multi_agent_env import MultiAgentTradingEnv, RISK_MANAGER, PORTFOLIO_MGR, TRADER
from env.reward import format_reward_func, alignment_reward_func, profit_reward_func
from training.grpo_verifiers_multiagent import (
    governance_reward_func_multiagent,
    risk_reward_func_multiagent,
)
from training.prompt_utils import (
    SYSTEM_PROMPT,
    build_prompt_multiagent,
    generate_pz_scenarios,
)
print("✅ All project imports successful")

## 2. PettingZoo Environment Demo

Quick sanity check — run the multi-agent AEC loop with rule-based policies.

In [ ]:
from training.train_multi_agent import (
    RuleRiskManagerPolicy,
    RulePortfolioManagerPolicy,
    RuleTraderPolicy,
    collect_rollout,
)

# Create environment instance
env = MultiAgentTradingEnv(difficulty="easy", max_steps=50)
policies = {
    RISK_MANAGER:  RuleRiskManagerPolicy(),
    PORTFOLIO_MGR: RulePortfolioManagerPolicy(),
    TRADER:        RuleTraderPolicy(),
}

# Run one episode
buffers, info = collect_rollout(env, policies, max_steps=50)

print("=" * 60)
print("  PettingZoo AEC Environment — Single Episode")
print("=" * 60)
print(f"  Trader steps:  {len(buffers[TRADER])}")
print(f"  RM steps:      {len(buffers[RISK_MANAGER])}")
print(f"  PM steps:      {len(buffers[PORTFOLIO_MGR])}")
print(f"  Final PnL:     {info.get('pnl_pct', 0):.2%}")
print(f"  Max Drawdown:  {info.get('max_drawdown', 0):.2%}")
print(f"  Sharpe:        {info.get('sharpe_ratio', 0):.3f}")
print(f"  Grade:         {info.get('grade', 0):.4f}")
print("=" * 60)
print("✅ Environment functional!")

## 3. Generate Training Scenarios

Run the PettingZoo env with rule-based RM/PM policies to collect realistic market scenarios. Each scenario contains:
- Base market observation (24 dims)
- Risk Manager constraints (`rm_size_limit`, `rm_allow_new`, `rm_force_reduce`)
- Portfolio Manager allocation (`pm_cap_alloc`, `pm_override`)

In [ ]:
import random
import numpy as np

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

# ─── Configuration ──────────────────────────────────────────────────────────
NUM_SCENARIOS  = 500          # Number of training prompts
DIFFICULTY     = "easy"       # Start easy for faster convergence
MAX_ENV_STEPS  = 100          # Steps per PZ episode for scenario generation

print(f"Generating {NUM_SCENARIOS} scenarios (difficulty={DIFFICULTY})...")
scenarios = generate_pz_scenarios(
    n=NUM_SCENARIOS,
    difficulty=DIFFICULTY,
    max_env_steps=MAX_ENV_STEPS,
)
print(f"✅ Generated {len(scenarios)} scenarios")

# Preview one scenario
import json
print("\n── Sample Scenario ──")
print(json.dumps(scenarios[0], indent=2, default=str))

In [ ]:
# ─── Build prompts and HF Dataset ───────────────────────────────────────────
from datasets import Dataset

prompts = [{"prompt": build_prompt_multiagent(sc)} for sc in scenarios]
dataset = Dataset.from_list(prompts)

print(f"✅ Dataset ready: {len(dataset)} prompts")
print("\n── Sample Prompt (first 500 chars) ──")
print(dataset[0]["prompt"][:500])

## 4. Load Model with Unsloth

Load Qwen 2.5 1.5B with 4-bit quantization and apply LoRA adapters for parameter-efficient training.

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# Patch TRL for 2x faster GRPO
PatchFastRL("GRPO", "unsloth")

# ─── Model Configuration ────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,         # Auto-detect (bf16 if supported)
    load_in_4bit=True,  # 4-bit quantization via bitsandbytes
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. GRPO Training

Train using Group Relative Policy Optimization with **5 independent reward verifiers**:

| Verifier | What it checks | Max Score |
|---|---|---|
| `format_reward_func` | `<thought>` + `<action>` tags, JSON validity | 1.0 |
| `alignment_reward_func` | Reasoning matches market signals (anti-hallucination) | 1.0 |
| `risk_reward_func_multiagent` | Compliance with RM size limit | 1.0 |
| `profit_reward_func` | Directional correctness vs. price trend | 1.0 |
| `governance_reward_func_multiagent` | Full RM + PM governance compliance | 1.0 |

In [ ]:
import inspect
from trl.trainer.grpo_config import GRPOConfig
from trl.trainer.grpo_trainer import GRPOTrainer

# ─── Training Hyperparameters ────────────────────────────────────────────────
OUTPUT_DIR         = "./grpo_output"
MAX_STEPS          = 250        # Training steps (increase for longer training)
BATCH_SIZE         = 4          # Per-device batch size
GRAD_ACCUM         = 2          # Gradient accumulation steps
NUM_GENERATIONS    = 4          # GRPO generations per prompt
LEARNING_RATE      = 5e-5
MAX_PROMPT_LENGTH  = 768
MAX_COMPLETION_LEN = 200
SAVE_STEPS         = 50
LOGGING_STEPS      = 1

# ─── GRPO Config ────────────────────────────────────────────────────────────
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LEN,
    num_generations=NUM_GENERATIONS,
    report_to="none",
)

# ─── Reward Functions ────────────────────────────────────────────────────────
reward_funcs = [
    format_reward_func,
    alignment_reward_func,
    risk_reward_func_multiagent,
    profit_reward_func,
    governance_reward_func_multiagent,
]

# ─── Build Trainer (handles TRL version differences) ─────────────────────────
trainer_kwargs = {
    "model": model,
    "reward_funcs": reward_funcs,
    "args": training_args,
    "train_dataset": dataset,
}

# TRL version compatibility: older = "tokenizer", newer = "processing_class"
sig = inspect.signature(GRPOTrainer.__init__)
if "processing_class" in sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = GRPOTrainer(**trainer_kwargs)

print(f"✅ GRPOTrainer initialized")
print(f"   Steps: {MAX_STEPS}  |  Batch: {BATCH_SIZE}  |  Generations: {NUM_GENERATIONS}")
print(f"   Reward functions: {len(reward_funcs)}")

In [ ]:
# ─── Run Training ───────────────────────────────────────────────────────────
print("🚀 Starting GRPO training...")
print(f"   This may take 30-60+ minutes on a T4 GPU.")
print()

trainer.train()

print("\n✅ Training complete!")

## 6. Training Results & Plots

Extract loss and reward curves from the training history and generate publication-ready plots.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})

# ─── Extract metrics from trainer log history ────────────────────────────────
history = trainer.state.log_history
steps   = [x["step"] for x in history if "loss" in x]
losses  = [x["loss"] for x in history if "loss" in x]
rewards = [x["reward"] for x in history if "reward" in x]
r_steps = [x["step"] for x in history if "reward" in x]

print(f"Collected {len(losses)} loss entries, {len(rewards)} reward entries")

# ─── Plot 1: Loss Curve ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(steps, losses, color="#e74c3c", linewidth=1.5, alpha=0.7, label="Raw")
# Smoothed line
if len(losses) > 10:
    window = min(20, len(losses) // 3)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    ax1.plot(steps[window-1:], smoothed, color="#c0392b", linewidth=2.5, label=f"MA-{window}")
ax1.set_xlabel("Training Step")
ax1.set_ylabel("Loss")
ax1.set_title("GRPO Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# ─── Plot 2: Reward Curve ────────────────────────────────────────────────────
ax2.plot(r_steps, rewards, color="#27ae60", linewidth=1.5, alpha=0.7, label="Raw")
if len(rewards) > 10:
    window = min(20, len(rewards) // 3)
    smoothed = np.convolve(rewards, np.ones(window)/window, mode="valid")
    ax2.plot(r_steps[window-1:], smoothed, color="#1e8449", linewidth=2.5, label=f"MA-{window}")
ax2.set_xlabel("Training Step")
ax2.set_ylabel("Mean Reward")
ax2.set_title("GRPO Mean Reward (5 Verifiers)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("QuantHive Multi-Agent GRPO Training — Qwen 2.5 1.5B", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", bbox_inches="tight")
plt.show()
print("✅ Saved training_curves.png")

In [ ]:
# ─── Save training metrics to JSON ──────────────────────────────────────────
metrics = {
    "losses": losses,
    "rewards": rewards,
    "final_loss": losses[-1] if losses else None,
    "final_reward": rewards[-1] if rewards else None,
    "best_reward": max(rewards) if rewards else None,
    "total_steps": MAX_STEPS,
    "model": MODEL_NAME,
    "num_scenarios": NUM_SCENARIOS,
}

with open("training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("📊 Training Summary:")
print(f"   Final Loss:   {metrics['final_loss']:.4f}" if metrics['final_loss'] else "   No loss data")
print(f"   Final Reward:  {metrics['final_reward']:.4f}" if metrics['final_reward'] else "   No reward data")
print(f"   Best Reward:   {metrics['best_reward']:.4f}" if metrics['best_reward'] else "   No reward data")

## 7. Save Model

Save the trained LoRA adapters. Optionally push to Hugging Face Hub.

In [ ]:
# ─── Save LoRA adapters locally ─────────────────────────────────────────────
ADAPTER_DIR = "./quanthive-trader-grpo-lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapters saved to {ADAPTER_DIR}")

# Optional: Save full merged model (16-bit) — uses more disk
# model.save_pretrained_merged("./quanthive-merged-16bit", tokenizer, save_method="merged_16bit")

In [ ]:
# ─── (Optional) Push to Hugging Face Hub ────────────────────────────────────
# Uncomment and set your token + repo to push adapters to HF

# HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # or use huggingface-cli login
# HF_REPO  = "ARKAISW/quanthive-trader-grpo-lora"
#
# model.push_to_hub(HF_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
# print(f"✅ Pushed to https://huggingface.co/{HF_REPO}")

## 8. Inference Test — Before vs After

Compare the trained model's outputs against the base model to demonstrate learning.

In [ ]:
# ─── Generate a test scenario ────────────────────────────────────────────────
test_scenarios = generate_pz_scenarios(n=3, difficulty="easy")
test_prompt = build_prompt_multiagent(test_scenarios[0])

# Switch to inference mode
FastLanguageModel.for_inference(model)

# Tokenize
inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_PROMPT_LENGTH,
).to("cuda")

# Generate
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_COMPLETION_LEN,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

# Decode only the generated part
generated = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=" * 60)
print("  🧪 Trained Model Inference")
print("=" * 60)
print(f"\nGovernance context:")
print(f"  RM size_limit:  {test_scenarios[0]['rm_size_limit']:.3f}")
print(f"  PM cap_alloc:   {test_scenarios[0]['pm_cap_alloc']:.3f}")
print(f"\nModel output:")
print(generated)
print("=" * 60)

In [ ]:
# ─── Score the generated output with all 5 reward verifiers ──────────────────
test_prompts     = [test_prompt]
test_completions = [generated]

scores = {
    "format":      format_reward_func(test_prompts, test_completions)[0],
    "alignment":   alignment_reward_func(test_prompts, test_completions)[0],
    "risk":        risk_reward_func_multiagent(test_prompts, test_completions)[0],
    "profit":      profit_reward_func(test_prompts, test_completions)[0],
    "governance":  governance_reward_func_multiagent(test_prompts, test_completions)[0],
}

print("\n📊 Reward Verifier Scores:")
print("-" * 40)
for name, score in scores.items():
    bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    print(f"  {name:15s}  {bar}  {score:.2f}")
print("-" * 40)
print(f"  {'TOTAL':15s}  {' '*20}  {sum(scores.values()):.2f} / 5.00")

## 9. Baseline Comparison

Run the same scenarios through the environment with the trained model vs. random baseline to demonstrate improvement.

In [ ]:
# ─── Score multiple scenarios: Trained vs Random ──────────────────────────────
N_EVAL = 20
eval_scenarios = generate_pz_scenarios(n=N_EVAL, difficulty="easy")

trained_scores = {k: [] for k in ["format", "alignment", "risk", "profit", "governance"]}
random_scores  = {k: [] for k in ["format", "alignment", "risk", "profit", "governance"]}

print(f"Evaluating {N_EVAL} scenarios...")

for i, sc in enumerate(eval_scenarios):
    prompt = build_prompt_multiagent(sc)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to("cuda")

    # Trained model generation
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_COMPLETION_LEN, do_sample=True, temperature=0.7, top_p=0.9)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Random baseline: random action string
    rand_completion = f'<thought>Random trade decision.</thought>\n<action>{{"direction": {random.choice([0,1,2])}, "size": {random.uniform(0.01, 0.9):.2f}, "sl": 0, "tp": 0}}</action>'

    # Score both
    for key, func in [("format", format_reward_func), ("alignment", alignment_reward_func),
                      ("risk", risk_reward_func_multiagent), ("profit", profit_reward_func),
                      ("governance", governance_reward_func_multiagent)]:
        trained_scores[key].append(func([prompt], [gen])[0])
        random_scores[key].append(func([prompt], [rand_completion])[0])

    if (i + 1) % 5 == 0:
        print(f"  [{i+1}/{N_EVAL}] done")

print("✅ Evaluation complete")

In [ ]:
# ─── Plot: Trained vs Random Baseline ────────────────────────────────────────
categories = list(trained_scores.keys())
trained_means = [np.mean(trained_scores[k]) for k in categories]
random_means  = [np.mean(random_scores[k]) for k in categories]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, random_means,  width, label="Random Baseline", color="#e74c3c", alpha=0.8)
bars2 = ax.bar(x + width/2, trained_means, width, label="GRPO-Trained",    color="#27ae60", alpha=0.8)

ax.set_xlabel("Reward Verifier")
ax.set_ylabel("Mean Score")
ax.set_title("QuantHive: Trained Agent vs Random Baseline")
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in categories])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("baseline_comparison.png", bbox_inches="tight")
plt.show()

# Summary table
print("\n📊 Comparison Summary:")
print(f"{'Verifier':15s}  {'Random':>8s}  {'Trained':>8s}  {'Δ':>8s}")
print("-" * 45)
for cat, rm, tm in zip(categories, random_means, trained_means):
    delta = tm - rm
    print(f"{cat:15s}  {rm:8.3f}  {tm:8.3f}  {delta:+8.3f}")
print("-" * 45)
print(f"{'TOTAL':15s}  {sum(random_means):8.3f}  {sum(trained_means):8.3f}  {sum(trained_means)-sum(random_means):+8.3f}")

## ✅ Done!

**Files generated:**
- `training_curves.png` — Loss and reward curves
- `baseline_comparison.png` — Trained vs. random baseline
- `training_metrics.json` — Raw metrics
- `quanthive-trader-grpo-lora/` — LoRA adapter weights

**Next steps:**
1. Download `training_curves.png` and `baseline_comparison.png` and commit to your repo's `plots/` directory
2. Push LoRA adapters to Hugging Face Hub (uncomment the push cell above)
3. For longer training runs with more compute, see the HF Jobs guide in the README

---

**GitHub:** [ARKAISW/multi-agent-trading-env](https://github.com/ARKAISW/multi-agent-trading-env)  
**HF Space:** [ARKAISW/QuantHive](https://huggingface.co/spaces/ARKAISW/QuantHive)